# SRE Agent Blueprint: Evidence-Grounded RCA with Constrained Agentic Refinement

This notebook is a **compact blueprint** for an AI-native SRE / AIOps RCA system. 

## What this notebook shows
1. A production-minded RCA architecture
2. A compact incident context model
3. Deterministic scoring and ambiguity detection
4. A constrained agentic refinement loop using mock MCP / RAG calls
5. Trial simulations for two ambiguous incident scenarios
6. The operator-facing value added by an LLM reasoning layer

## Design principles
- **Deterministic system computes**
- **LLM synthesizes and communicates**
- **Agentic refinement activates only under ambiguity**
- **Tool use is bounded, cheap-first, and evidence-driven**

## Table of Contents

1. [Architecture Overview](#1.-Architecture-Overview)
2. [Compact Incident Context Model](#2.-Compact-Incident-Context-Model)
3. [Approved Tool Set (Mock MCP / RAG)](#3.-Approved-Tool-Set-(Mock-MCP-/-RAG))
4. [MCP Tool Catalog](#4.-MCP-Tool-Catalog)
5. [Deterministic Scoring and Ambiguity Detection](#5.-Deterministic-Scoring-and-Ambiguity-Detection)
6. [Agent Policy](#6.-Agent-Policy)
7. [Deterministic Context Updates](#7.-Deterministic-Context-Updates)
8. [Bounded Agent Loop](#8.-Bounded-Agent-Loop)
9. [Scenario A: Retry Storm vs DB Bottleneck](#9.-Scenario-A:-Retry-Storm-vs-DB-Bottleneck)
10. [Scenario B: Deployment Regression vs Known Issue](#10.-Scenario-B:-Deployment-Regression-vs-Known-Issue)
11. [Simple Learning Loop](#11.-Simple-Learning-Loop)
12. [What the LLM Adds](#12.-What-the-LLM-Adds)
13. [Final Takeaway](#13.-Final-Takeaway)

## 1. Architecture Overview

```text
Signals (Traces, Metrics, Logs, HAR, JFR)
        ↓
Feature normalization + deterministic RCA
        ↓
Hypotheses + evidence scores + confidence
        ↓
Ambiguity detection
        ↓
If strong → LLM synthesis
If ambiguous → Constrained agentic refinement
        ↓
Fetch targeted evidence (MCP / RAG)
        ↓
Update context + re-score
        ↓
LLM synthesis layer
        ↓
Operator-facing RCA + mitigation guidance
        ↓
Post-incident learning
        ↓
Update scoring, thresholds, and retrieval priors
```

## Key idea
The system does **not** ask the LLM to discover root cause from raw telemetry.

Instead:
- the deterministic layer narrows the problem
- the agentic layer fetches missing context when needed
- the LLM turns refined evidence into an actionable RCA narrative
- the learning loop uses confirmed outcomes to improve future scoring and retrieval

In [1]:
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional, Tuple

## 2. Compact Incident Context Model

The agent starts from a structured incident context, not from raw telemetry. This keeps the design explainable and reproducible.

In [2]:
@dataclass
class EvidenceItem:
    evidence_id: str
    source: str
    description: str
    score: float

@dataclass
class Hypothesis:
    name: str
    evidence_score: float
    supporting_evidence: List[str] = field(default_factory=list)

@dataclass
class IncidentContext:
    incident_id: str
    service: str
    business_service: str
    severity: str
    scenario: str
    signals: Dict[str, Any]
    evidence: Dict[str, EvidenceItem]
    hypotheses: Dict[str, Hypothesis]
    confidence: float
    notes: List[str] = field(default_factory=list)

def summarize_context(ctx: IncidentContext):
    print(f"Scenario: {ctx.scenario}")
    print(f"Incident: {ctx.incident_id}")
    print(f"Service: {ctx.service} | Business service: {ctx.business_service} | Severity: {ctx.severity}")
    print(f"Confidence: {ctx.confidence:.2f}")
    print("\nHypotheses:")
    for h in sorted(ctx.hypotheses.values(), key=lambda x: x.evidence_score, reverse=True):
        print(f"  - {h.name}: score={h.evidence_score:.2f}, evidence={h.supporting_evidence}")

## 3. Approved Tool Set (Mock MCP / RAG)

These calls are mocked, but the structure mirrors a realistic system:
- MCP-style calls for deployments, config, and DB health
- RAG-style retrieval for known bugs and similar incidents

The purpose is to show **decision flow**, not external integration plumbing.

In [3]:
def get_recent_deployments(service: str, scenario: str) -> Dict[str, Any]:
    if scenario == "deployment_vs_known_issue":
        return {
            "tool": "get_recent_deployments",
            "service": service,
            "result": {
                "deployed_at": "2026-03-20T09:15:00Z",
                "version": "8.4.1",
                "change_note": "New profile enrichment logic added to response path"
            }
        }
    return {
        "tool": "get_recent_deployments",
        "service": service,
        "result": {
            "deployed_at": "2026-03-18T13:40:00Z",
            "version": "3.14.7",
            "change_note": "Updated retry policy defaults"
        }
    }

def get_config_diff(service: str, scenario: str) -> Dict[str, Any]:
    if scenario == "retry_vs_db":
        return {
            "tool": "get_config_diff",
            "service": service,
            "result": {
                "retry.maxAttempts": {"previous": 2, "current": 5},
                "retry.backoffMs": {"previous": 200, "current": 50}
            }
        }
    return {
        "tool": "get_config_diff",
        "service": service,
        "result": {}
    }

def get_db_health(db_name: str, scenario: str) -> Dict[str, Any]:
    if scenario == "retry_vs_db":
        return {
            "tool": "get_db_health",
            "db": db_name,
            "result": {
                "cpu_pct": 54,
                "storage_latency_ms": 11,
                "connection_pressure": "moderate",
                "global_saturation": False
            }
        }
    return {
        "tool": "get_db_health",
        "db": db_name,
        "result": {
            "cpu_pct": 48,
            "storage_latency_ms": 9,
            "connection_pressure": "low",
            "global_saturation": False
        }
    }

def query_knowledge_base(service: str, signature: str, scenario: str) -> Dict[str, Any]:
    if scenario == "retry_vs_db":
        return {
            "tool": "query_knowledge_base",
            "service": service,
            "signature": signature,
            "result": {
                "match_found": True,
                "bug_id": "BUG-8421",
                "match_score": 0.88,
                "summary": "Aggressive retry policy can amplify downstream latency into partial outages.",
                "patch_available": True,
                "workaround": "Reduce retries; restore exponential backoff."
            }
        }
    if scenario == "deployment_vs_known_issue":
        return {
            "tool": "query_knowledge_base",
            "service": service,
            "signature": signature,
            "result": {
                "match_found": True,
                "bug_id": "BUG-9912",
                "match_score": 0.91,
                "summary": "Known cache invalidation bug in profile service can cause stale reads and repeated re-fetch loops.",
                "patch_available": True,
                "workaround": "Apply hotfix or disable enrichment feature flag."
            }
        }
    return {"tool": "query_knowledge_base", "service": service, "signature": signature, "result": {"match_found": False}}

def retrieve_similar_incidents(query: str, scenario: str) -> Dict[str, Any]:
    if scenario == "retry_vs_db":
        return {
            "tool": "retrieve_similar_incidents",
            "query": query,
            "result": [
                {
                    "incident_id": "INC-2025-087",
                    "similarity": 0.81,
                    "title": "Retry amplification after dependency slowdown",
                    "remediation": ["reduce retries", "enable circuit breaking", "validate downstream health"]
                }
            ]
        }
    if scenario == "deployment_vs_known_issue":
        return {
            "tool": "retrieve_similar_incidents",
            "query": query,
            "result": [
                {
                    "incident_id": "INC-2025-144",
                    "similarity": 0.79,
                    "title": "Profile enrichment regression after deploy",
                    "remediation": ["rollback deployment", "disable feature flag", "inspect cache behavior"]
                }
            ]
        }
    return {"tool": "retrieve_similar_incidents", "query": query, "result": []}

## 4. MCP Tool Catalog

In a production design, MCP is a strong abstraction for exposing **retrieval and action tools** to the RCA system.  
The LLM or policy layer can choose which tool to call next, while the system still enforces:
- loop bounds
- permissions
- cost / latency budgets
- evidence gating

A compact tool catalog for this blueprint could include four categories:

### A. Context and Change Retrieval
- `get_deployment_history(service)`
- `get_config_diff(service)`
- `get_feature_flags(service)`
- `get_release_notes(service)`

### B. Runtime and Topology Retrieval
- `get_db_health(db)`
- `get_service_topology(service)`
- `get_blast_radius(service)`
- `get_dependency_health(service)`

### C. Knowledge and Incident Retrieval
- `query_knowledge_base(signature)`
- `retrieve_similar_incidents(query)`
- `get_runbook(service, symptom)`
- `get_known_bugs(service, version)`

### D. Operational Actions and Coordination
- `create_incident_update(ticket_or_channel)`
- `rollback_service(service, version)`
- `toggle_feature_flag(flag_name, state)`
- `scale_service(service, replica_count)`

## Design intent
Not every tool should be called in every incident. The system should:
- prefetch only the cheapest, highest-value baseline context
- use agentic refinement only when ambiguity remains
- reserve operational actions for explicit approval or tightly controlled automation

In [4]:
MCP_TOOL_CATALOG = {
    "context_change_retrieval": [
        "get_deployment_history(service)",
        "get_config_diff(service)",
        "get_feature_flags(service)",
        "get_release_notes(service)",
    ],
    "runtime_topology_retrieval": [
        "get_db_health(db)",
        "get_service_topology(service)",
        "get_blast_radius(service)",
        "get_dependency_health(service)",
    ],
    "knowledge_incident_retrieval": [
        "query_knowledge_base(signature)",
        "retrieve_similar_incidents(query)",
        "get_runbook(service, symptom)",
        "get_known_bugs(service, version)",
    ],
    "operational_actions_coordination": [
        "create_incident_update(ticket_or_channel)",
        "rollback_service(service, version)",
        "toggle_feature_flag(flag_name, state)",
        "scale_service(service, replica_count)",
    ],
}

MCP_TOOL_CATALOG

{'context_change_retrieval': ['get_deployment_history(service)',
  'get_config_diff(service)',
  'get_feature_flags(service)',
  'get_release_notes(service)'],
 'runtime_topology_retrieval': ['get_db_health(db)',
  'get_service_topology(service)',
  'get_blast_radius(service)',
  'get_dependency_health(service)'],
 'knowledge_incident_retrieval': ['query_knowledge_base(signature)',
  'retrieve_similar_incidents(query)',
  'get_runbook(service, symptom)',
  'get_known_bugs(service, version)'],
 'operational_actions_coordination': ['create_incident_update(ticket_or_channel)',
  'rollback_service(service, version)',
  'toggle_feature_flag(flag_name, state)',
  'scale_service(service, replica_count)']}

### How this fits the blueprint

In this notebook:
- the **policy layer** chooses the next best tool
- the **system layer** controls whether another step is allowed
- the **LLM** does not execute open-ended behavior; it works through the approved tool catalog

This makes the design more concrete while preserving a controlled, production-minded architecture.

## 5. Deterministic Scoring and Ambiguity Detection

The scoring model is intentionally simple:
- rank candidate hypotheses by evidence score
- compute confidence from both **score strength** and **separation**
- trigger refinement only when ambiguity remains

In [5]:
def top_two_hypotheses(ctx: IncidentContext) -> Tuple[Hypothesis, Optional[Hypothesis]]:
    ordered = sorted(ctx.hypotheses.values(), key=lambda x: x.evidence_score, reverse=True)
    if len(ordered) == 1:
        return ordered[0], None
    return ordered[0], ordered[1]

def recompute_confidence(ctx: IncidentContext) -> float:
    h1, h2 = top_two_hypotheses(ctx)
    if h2 is None:
        ctx.confidence = max(0.0, min(1.0, h1.evidence_score))
        return ctx.confidence
    total = sum(max(h.evidence_score, 0.0) for h in ctx.hypotheses.values())
    normalized_top = h1.evidence_score / total if total > 0 else 0.0
    gap = max(h1.evidence_score - h2.evidence_score, 0.0)
    ctx.confidence = min(max(0.55 * normalized_top + 0.45 * gap, 0.0), 1.0)
    return ctx.confidence

def ambiguity_trigger(ctx: IncidentContext, conf_threshold: float = 0.35, gap_threshold: float = 0.10) -> bool:
    h1, h2 = top_two_hypotheses(ctx)
    if h2 is None:
        return False
    gap = h1.evidence_score - h2.evidence_score
    return ctx.confidence < conf_threshold or gap < gap_threshold

## 6. Agent Policy

The agent has a **small action set** and uses a simple policy:
- prefer cheap, high-value evidence first
- choose evidence that best separates the top competing hypotheses
- stop early once confidence or score gap improves enough

This is a **bounded refinement loop**, not an open-ended autonomous agent.

In [6]:
def choose_next_action(ctx: IncidentContext) -> str:
    checked = {e.source for e in ctx.evidence.values()}

    if ctx.scenario == "retry_vs_db":
        if "mcp_config" not in checked:
            return "get_config_diff"
        if "mcp_db" not in checked:
            return "get_db_health"
        return "query_knowledge_base"

    if ctx.scenario == "deployment_vs_known_issue":
        if "mcp_deploy" not in checked:
            return "get_recent_deployments"
        if "rag_kb" not in checked:
            return "query_knowledge_base"
        return "retrieve_similar_incidents"

    return "retrieve_similar_incidents"

## 7. Deterministic Context Updates

Tool outputs update the structured context and re-score hypotheses. The point is to keep the source of truth deterministic and inspectable.

In [7]:
def apply_tool_result(ctx: IncidentContext, action: str, result: Dict[str, Any]) -> None:
    if action == "get_config_diff":
        changes = result["result"]
        aggressive_retry = (
            changes.get("retry.maxAttempts", {}).get("current", 0) >= 5 and
            changes.get("retry.backoffMs", {}).get("current", 9999) <= 50
        )
        if aggressive_retry:
            ctx.evidence["MCP_CFG"] = EvidenceItem("MCP_CFG", "mcp_config", "Retry policy recently made more aggressive", 0.95)
            ctx.hypotheses["RETRY_STORM"].evidence_score += 0.10
            ctx.notes.append("Config diff strengthened RETRY_STORM hypothesis.")

    elif action == "get_db_health":
        db = result["result"]
        if db["global_saturation"] is False and db["cpu_pct"] < 70:
            ctx.evidence["MCP_DB"] = EvidenceItem("MCP_DB", "mcp_db", "DB not globally saturated; pressure likely secondary/localized", 0.85)
            ctx.hypotheses["DATABASE_BOTTLENECK"].evidence_score -= 0.05
            ctx.hypotheses["RETRY_STORM"].evidence_score += 0.03
            ctx.notes.append("DB health weakens primary DB bottleneck interpretation.")

    elif action == "query_knowledge_base":
        kb = result["result"]
        if kb["match_found"]:
            ctx.evidence["KB1"] = EvidenceItem("KB1", "rag_kb", kb["summary"], kb["match_score"])
            if ctx.scenario == "retry_vs_db":
                ctx.hypotheses["RETRY_STORM"].evidence_score += 0.06
                ctx.notes.append("Knowledge base match supports retry amplification pattern.")
            elif ctx.scenario == "deployment_vs_known_issue":
                if "KNOWN_ISSUE" in ctx.hypotheses:
                    ctx.hypotheses["KNOWN_ISSUE"].evidence_score += 0.10
                ctx.notes.append("Knowledge base match supports known issue interpretation.")

    elif action == "retrieve_similar_incidents":
        sims = result["result"]
        if sims:
            top = sims[0]
            ctx.evidence["RAG1"] = EvidenceItem("RAG1", "rag_incidents", top["title"], top["similarity"])
            if ctx.scenario == "deployment_vs_known_issue":
                if "DEPLOYMENT_REGRESSION" in ctx.hypotheses:
                    ctx.hypotheses["DEPLOYMENT_REGRESSION"].evidence_score += 0.05
                ctx.notes.append("Similar incident retrieval supports deployment-related explanation.")
            else:
                ctx.hypotheses["RETRY_STORM"].evidence_score += 0.05
                ctx.notes.append("Similar incident retrieval supports RETRY_STORM.")

    elif action == "get_recent_deployments":
        dep = result["result"]
        ctx.evidence["MCP_DEPLOY"] = EvidenceItem("MCP_DEPLOY", "mcp_deploy", f"Recent deployment: {dep['change_note']}", 0.80)
        if "DEPLOYMENT_REGRESSION" in ctx.hypotheses:
            ctx.hypotheses["DEPLOYMENT_REGRESSION"].evidence_score += 0.08
        ctx.notes.append("Recent deployment strengthens DEPLOYMENT_REGRESSION hypothesis.")

    recompute_confidence(ctx)

## 8. Bounded Agent Loop

In [8]:
def run_constrained_agent(ctx: IncidentContext, max_steps: int = 3,
                          conf_threshold: float = 0.35, gap_threshold: float = 0.10) -> IncidentContext:
    recompute_confidence(ctx)

    if not ambiguity_trigger(ctx, conf_threshold, gap_threshold):
        print("No refinement needed; deterministic RCA is already strong.")
        return ctx

    for step in range(1, max_steps + 1):
        h1, h2 = top_two_hypotheses(ctx)
        gap = h1.evidence_score - (h2.evidence_score if h2 else 0.0)
        print(f"Step {step}: {h1.name}={h1.evidence_score:.2f}, {h2.name if h2 else 'N/A'}={h2.evidence_score if h2 else 0:.2f}, confidence={ctx.confidence:.2f}, gap={gap:.2f}")

        if not ambiguity_trigger(ctx, conf_threshold, gap_threshold):
            print("Stopping: ambiguity resolved.")
            break

        action = choose_next_action(ctx)
        print("  Chosen action:", action)

        if action == "get_config_diff":
            result = get_config_diff(ctx.service, ctx.scenario)
        elif action == "get_db_health":
            result = get_db_health("inventory-db", ctx.scenario)
        elif action == "query_knowledge_base":
            result = query_knowledge_base(ctx.service, "incident signature", ctx.scenario)
        elif action == "retrieve_similar_incidents":
            result = retrieve_similar_incidents(f"{ctx.service} ambiguity", ctx.scenario)
        elif action == "get_recent_deployments":
            result = get_recent_deployments(ctx.service, ctx.scenario)
        else:
            raise ValueError(action)

        apply_tool_result(ctx, action, result)

    print("\nFinal refined state:")
    summarize_context(ctx)
    return ctx

## 9. Scenario A: Retry Storm vs DB Bottleneck

This scenario starts with a close call between:
- **DATABASE_BOTTLENECK**
- **RETRY_STORM**

The goal is to show how bounded refinement can distinguish **primary cause** from **secondary symptom**.

In [9]:
ctx_a = IncidentContext(
    incident_id="INC-2026-041",
    service="inventory-service",
    business_service="checkout",
    severity="SEV-2",
    scenario="retry_vs_db",
    signals={
        "metrics": {"db_pool_utilization_pct": 93, "db_wait_ms": 420, "error_rate_pct": 7.9},
        "logs": {"top_templates": ["timeout waiting for DB connection", "retry initiated"]},
        "traces": {"dominant_pattern": "inventory timeout path", "retry_count": 3.8},
        "jfr": {"thread_blocked_pct": 0.61, "dominant_wait_type": "mixed_db_and_io"}
    },
    evidence={
        "M1": EvidenceItem("M1", "metrics", "DB pool utilization is high", 0.90),
        "L1": EvidenceItem("L1", "logs", "Timeout waiting for DB connection", 0.82),
        "L2": EvidenceItem("L2", "logs", "Retry initiated", 0.78),
        "T1": EvidenceItem("T1", "traces", "Retry count elevated", 0.80),
        "J1": EvidenceItem("J1", "jfr", "Threads blocked with mixed DB/IO wait", 0.72),
    },
    hypotheses={
        "DATABASE_BOTTLENECK": Hypothesis("DATABASE_BOTTLENECK", 0.71, ["M1", "L1", "J1"]),
        "RETRY_STORM": Hypothesis("RETRY_STORM", 0.68, ["L2", "T1"]),
        "DEPENDENCY_LATENCY": Hypothesis("DEPENDENCY_LATENCY", 0.54, ["J1"]),
    },
    confidence=0.0
)

recompute_confidence(ctx_a)
summarize_context(ctx_a)
print("\nAmbiguity trigger:", ambiguity_trigger(ctx_a))

Scenario: retry_vs_db
Incident: INC-2026-041
Service: inventory-service | Business service: checkout | Severity: SEV-2
Confidence: 0.22

Hypotheses:
  - DATABASE_BOTTLENECK: score=0.71, evidence=['M1', 'L1', 'J1']
  - RETRY_STORM: score=0.68, evidence=['L2', 'T1']
  - DEPENDENCY_LATENCY: score=0.54, evidence=['J1']

Ambiguity trigger: True


In [10]:
ctx_a = run_constrained_agent(ctx_a)

Step 1: DATABASE_BOTTLENECK=0.71, RETRY_STORM=0.68, confidence=0.22, gap=0.03
  Chosen action: get_config_diff
Step 2: RETRY_STORM=0.78, DATABASE_BOTTLENECK=0.71, confidence=0.24, gap=0.07
  Chosen action: get_db_health
Step 3: RETRY_STORM=0.81, DATABASE_BOTTLENECK=0.66, confidence=0.29, gap=0.15
  Chosen action: query_knowledge_base

Final refined state:
Scenario: retry_vs_db
Incident: INC-2026-041
Service: inventory-service | Business service: checkout | Severity: SEV-2
Confidence: 0.33

Hypotheses:
  - RETRY_STORM: score=0.87, evidence=['L2', 'T1']
  - DATABASE_BOTTLENECK: score=0.66, evidence=['M1', 'L1', 'J1']
  - DEPENDENCY_LATENCY: score=0.54, evidence=['J1']


### Scenario A: Representative LLM Synthesis

Below is the kind of operator-facing RCA a model would produce **after** deterministic scoring and refinement.

In [11]:
llm_response_a = '''
A. Incident Summary
The most likely primary issue is a RETRY_STORM triggered by an overly aggressive retry policy, with localized DB pressure acting as a secondary effect rather than the primary bottleneck.

B. Observed Facts
- DB pool utilization and DB wait time are elevated.
- Logs show both DB timeout and retry initiation patterns.
- Traces show increased retry activity.
- JFR shows mixed DB and IO wait, not clear evidence of global DB saturation.
- Config diff shows retry.maxAttempts increased and backoff reduced.
- DB health indicates no broad DB saturation.
- Knowledge-base evidence matches retry amplification behavior.

C. Primary RCA
RETRY_STORM

D. Contributing Factors
- DATABASE_BOTTLENECK (secondary / localized pressure)
- DEPENDENCY_LATENCY (possible co-factor)

E. Why This Hypothesis Wins
Retry amplification is better supported because the strongest new evidence points to an amplification mechanism, while DB health weakens the interpretation that database capacity alone is the primary cause.

F. Immediate Mitigation
1. Restore safer retry settings.
2. Validate downstream dependency health.
3. Monitor DB pressure after retry reduction before scaling DB capacity.
4. If needed, temporarily reduce low-priority workloads.

G. Confidence and Uncertainty
Confidence is moderate because DB pressure is real, but current evidence indicates it is more likely a downstream consequence than the trigger.
'''.strip()

print(llm_response_a)

A. Incident Summary
The most likely primary issue is a RETRY_STORM triggered by an overly aggressive retry policy, with localized DB pressure acting as a secondary effect rather than the primary bottleneck.

B. Observed Facts
- DB pool utilization and DB wait time are elevated.
- Logs show both DB timeout and retry initiation patterns.
- Traces show increased retry activity.
- JFR shows mixed DB and IO wait, not clear evidence of global DB saturation.
- Config diff shows retry.maxAttempts increased and backoff reduced.
- DB health indicates no broad DB saturation.
- Knowledge-base evidence matches retry amplification behavior.

C. Primary RCA
RETRY_STORM

D. Contributing Factors
- DATABASE_BOTTLENECK (secondary / localized pressure)
- DEPENDENCY_LATENCY (possible co-factor)

E. Why This Hypothesis Wins
Retry amplification is better supported because the strongest new evidence points to an amplification mechanism, while DB health weakens the interpretation that database capacity alone i

## 10. Scenario B: Deployment Regression vs Known Issue

This scenario starts with a different ambiguity:
- **DEPLOYMENT_REGRESSION**
- **KNOWN_ISSUE**

The goal is to show that the same blueprint can handle another type of uncertainty without changing the overall design.

In [12]:
ctx_b = IncidentContext(
    incident_id="INC-2026-052",
    service="profile-service",
    business_service="user-profile",
    severity="SEV-2",
    scenario="deployment_vs_known_issue",
    signals={
        "metrics": {"latency_p95_ms": 3100, "error_rate_pct": 2.1, "cpu_pct": 46},
        "logs": {"top_templates": ["cache miss burst", "profile enrichment re-fetch triggered"]},
        "traces": {"dominant_pattern": "profile enrichment path", "fanout_growth": "moderate"},
        "jfr": {"thread_blocked_pct": 0.22, "dominant_wait_type": "cache_io"}
    },
    evidence={
        "M2": EvidenceItem("M2", "metrics", "Latency spike without major CPU pressure", 0.74),
        "L3": EvidenceItem("L3", "logs", "Repeated profile enrichment re-fetch", 0.76),
        "T2": EvidenceItem("T2", "traces", "Enrichment path expanded after recent behavior shift", 0.73),
        "J2": EvidenceItem("J2", "jfr", "Cache IO wait present but not severe", 0.60),
    },
    hypotheses={
        "DEPLOYMENT_REGRESSION": Hypothesis("DEPLOYMENT_REGRESSION", 0.69, ["M2", "T2"]),
        "KNOWN_ISSUE": Hypothesis("KNOWN_ISSUE", 0.66, ["L3"]),
        "DEPENDENCY_LATENCY": Hypothesis("DEPENDENCY_LATENCY", 0.49, ["J2"]),
    },
    confidence=0.0
)

recompute_confidence(ctx_b)
summarize_context(ctx_b)
print("\nAmbiguity trigger:", ambiguity_trigger(ctx_b))

Scenario: deployment_vs_known_issue
Incident: INC-2026-052
Service: profile-service | Business service: user-profile | Severity: SEV-2
Confidence: 0.22

Hypotheses:
  - DEPLOYMENT_REGRESSION: score=0.69, evidence=['M2', 'T2']
  - KNOWN_ISSUE: score=0.66, evidence=['L3']
  - DEPENDENCY_LATENCY: score=0.49, evidence=['J2']

Ambiguity trigger: True


In [13]:
ctx_b = run_constrained_agent(ctx_b)

Step 1: DEPLOYMENT_REGRESSION=0.69, KNOWN_ISSUE=0.66, confidence=0.22, gap=0.03
  Chosen action: get_recent_deployments
Step 2: DEPLOYMENT_REGRESSION=0.77, KNOWN_ISSUE=0.66, confidence=0.27, gap=0.11
  Chosen action: query_knowledge_base
Step 3: DEPLOYMENT_REGRESSION=0.77, KNOWN_ISSUE=0.76, confidence=0.21, gap=0.01
  Chosen action: retrieve_similar_incidents

Final refined state:
Scenario: deployment_vs_known_issue
Incident: INC-2026-052
Service: profile-service | Business service: user-profile | Severity: SEV-2
Confidence: 0.24

Hypotheses:
  - DEPLOYMENT_REGRESSION: score=0.82, evidence=['M2', 'T2']
  - KNOWN_ISSUE: score=0.76, evidence=['L3']
  - DEPENDENCY_LATENCY: score=0.49, evidence=['J2']


### Scenario B: Representative LLM Synthesis

In [14]:
llm_response_b = '''
A. Incident Summary
The leading explanation is a KNOWN_ISSUE in profile enrichment and cache invalidation behavior, with a recent deployment acting as the likely exposure trigger rather than the sole root cause.

B. Observed Facts
- Latency increased significantly without corresponding CPU saturation.
- Logs show repeated enrichment re-fetch behavior.
- Traces show expansion in the profile enrichment path.
- A recent deployment introduced enrichment logic changes.
- Knowledge-base lookup strongly matches a known cache invalidation bug with an available workaround.

C. Primary RCA
KNOWN_ISSUE

D. Contributing Factors
- DEPLOYMENT_REGRESSION (likely exposure trigger)
- DEPENDENCY_LATENCY (minor supporting factor)

E. Why This Hypothesis Wins
The deployment explains timing, but the knowledge-base match provides a more specific mechanism that aligns with logs and trace behavior. That makes the known issue a stronger root cause explanation than treating the deploy itself as the full cause.

F. Immediate Mitigation
1. Disable the enrichment feature flag or apply the known workaround.
2. If user impact remains high, roll back the recent deployment.
3. Validate cache behavior after mitigation.
4. Schedule patch application.

G. Confidence and Uncertainty
Confidence is moderate-to-high because the KB match is strong and aligns with observed behavior, though deployment timing still matters as an enabling condition.
'''.strip()

print(llm_response_b)

A. Incident Summary
The leading explanation is a KNOWN_ISSUE in profile enrichment and cache invalidation behavior, with a recent deployment acting as the likely exposure trigger rather than the sole root cause.

B. Observed Facts
- Latency increased significantly without corresponding CPU saturation.
- Logs show repeated enrichment re-fetch behavior.
- Traces show expansion in the profile enrichment path.
- A recent deployment introduced enrichment logic changes.
- Knowledge-base lookup strongly matches a known cache invalidation bug with an available workaround.

C. Primary RCA
KNOWN_ISSUE

D. Contributing Factors
- DEPLOYMENT_REGRESSION (likely exposure trigger)
- DEPENDENCY_LATENCY (minor supporting factor)

E. Why This Hypothesis Wins
The deployment explains timing, but the knowledge-base match provides a more specific mechanism that aligns with logs and trace behavior. That makes the known issue a stronger root cause explanation than treating the deploy itself as the full cause.


## 11. Simple Learning Loop

A production RCA system should not remain static after each incident. Even a lightweight design can improve over time by comparing:
- predicted RCA
- reported confidence
- actual post-incident ground truth
- successful mitigation steps

In this compact blueprint, the learning loop is intentionally simple:

1. Capture confirmed root cause and effective remediation
2. Compare predicted hypothesis vs actual outcome
3. Apply small adjustments to:
   - feature weights / hypothesis priors
   - ambiguity thresholds
   - retrieval priors for similar incidents
4. Store the resolved incident as a future retrieval artifact

This keeps the notebook readable while still showing how the system evolves beyond one-off RCA.

In [15]:
def apply_post_incident_feedback(ctx: IncidentContext, actual_root_cause: str, successful_actions: list[str]) -> dict:
    """Simple scaffold illustrating how post-incident feedback updates the system."""
    predicted = sorted(ctx.hypotheses.values(), key=lambda x: x.evidence_score, reverse=True)[0].name

    learning_update = {
        "predicted_root_cause": predicted,
        "actual_root_cause": actual_root_cause,
        "was_correct": predicted == actual_root_cause,
        "successful_actions": successful_actions,
        "adjustments": []
    }

    if predicted != actual_root_cause:
        learning_update["adjustments"].append(
            f"Reduce prior weight for {predicted} in similar future scenarios"
        )
        if actual_root_cause in ctx.hypotheses:
            learning_update["adjustments"].append(
                f"Increase weight for evidence patterns supporting {actual_root_cause}"
            )
    else:
        learning_update["adjustments"].append(
            f"Increase confidence calibration for {actual_root_cause} when similar signals recur"
        )

    learning_update["adjustments"].append(
        "Store this incident as a retrieval artifact for future RAG lookups"
    )
    return learning_update

### Example Learning Update: Scenario A

After incident resolution, assume the postmortem confirms:
- **Actual root cause:** RETRY_STORM
- **Effective mitigation:** restore safe retry settings, validate dependency health

The feedback loop uses that outcome to reinforce the correct interpretation for similar future incidents.

In [16]:
learning_a = apply_post_incident_feedback(
    ctx_a,
    actual_root_cause="RETRY_STORM",
    successful_actions=[
        "Restore safe retry configuration",
        "Validate downstream dependency health"
    ]
)
learning_a

{'predicted_root_cause': 'RETRY_STORM',
 'actual_root_cause': 'RETRY_STORM',
 'was_correct': True,
 'successful_actions': ['Restore safe retry configuration',
  'Validate downstream dependency health'],
 'adjustments': ['Increase confidence calibration for RETRY_STORM when similar signals recur',
  'Store this incident as a retrieval artifact for future RAG lookups']}

### Example Learning Update: Scenario B

Assume the postmortem confirms:
- **Actual root cause:** KNOWN_ISSUE
- **Effective mitigation:** disable feature flag, apply workaround / patch

This creates a new high-value retrieval artifact and improves future ambiguity resolution between deployment timing and known defect patterns.

In [17]:
learning_b = apply_post_incident_feedback(
    ctx_b,
    actual_root_cause="KNOWN_ISSUE",
    successful_actions=[
        "Disable enrichment feature flag",
        "Apply known workaround / patch"
    ]
)
learning_b

{'predicted_root_cause': 'DEPLOYMENT_REGRESSION',
 'actual_root_cause': 'KNOWN_ISSUE',
 'was_correct': False,
 'successful_actions': ['Disable enrichment feature flag',
  'Apply known workaround / patch'],
 'adjustments': ['Reduce prior weight for DEPLOYMENT_REGRESSION in similar future scenarios',
  'Increase weight for evidence patterns supporting KNOWN_ISSUE',
  'Store this incident as a retrieval artifact for future RAG lookups']}

## 12. What the LLM Adds

The deterministic system already provides:
- candidate hypotheses
- evidence scores
- confidence
- refined context after MCP / RAG enrichment

The LLM layer adds:
- a coherent RCA narrative
- causal arbitration between close hypotheses
- explanation of why one cause wins
- mitigation sequencing
- concise uncertainty handling

This is the intended division of labor:

- **System computes**
- **LLM synthesizes and communicates**

## 13. Final Takeaway

This notebook illustrates a compact, production-minded blueprint for **evidence-grounded RCA with constrained agentic refinement**.

The important design choices are:
- deterministic scoring first
- refinement only under ambiguity
- bounded tool use
- LLM used for synthesis rather than raw discovery

That makes the system:
- easier to reason about
- cheaper to operate
- safer to evolve
- easier to explain in architecture reviews